## Imports

In [ ]:
# PyTorch & Numpy imports
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import to_tensor

# Dataset imports
from PIL import Image
from pathlib import Path

# Visualization import
import matplotlib.pyplot as plt

## The `ImageDirectoryDataset` Class

This is the `Dataset` class we went through a couple lectures ago. It loads labeled images from disk and makes them available for training.

In [ ]:
class ImageDirectoryDataset(Dataset):
    EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"}

    def __init__(self, directory):
        root = Path(directory).expanduser()
        if not root.is_dir():
            raise NotADirectoryError(root)

        # Sorting makes label assignments reproducible.
        class_dirs = sorted(
            (p for p in root.iterdir() if p.is_dir()),
            key=lambda p: p.name,
        )
        self.classes = [p.name for p in class_dirs]
        self.class_to_idx = {
            name: index for index, name in enumerate(self.classes)
        }

        self.samples = [
            (path, self.class_to_idx[class_dir.name])
            for class_dir in class_dirs
            for path in sorted(class_dir.rglob("*"))
            if path.is_file() and path.suffix.lower() in self.EXTENSIONS
        ]

        if not self.samples:
            raise ValueError(f"No images found in class subdirectories of {root}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        with Image.open(path) as image:
            # if we had color data, we would use this line instead of plain to_tensor:
            #image = to_tensor(image.convert("RGB"))
            image = to_tensor(image)
        return image, label

### Loading our data from disk

In [ ]:
train_ds = ImageDirectoryDataset("./data/binary-mnist/train")
test_ds = ImageDirectoryDataset("./data/binary-mnist/test")

### Visualizing our data

In [ ]:
image, label = train_ds[0]
plt.figure(figsize=(3, 3))
plt.imshow(image.squeeze(), cmap="gray")
plt.title(f"Digit: {label}")
plt.axis("off")
plt.show()

## Creating the `DataLoader`s

In [ ]:
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=True)

## The Model

In [ ]:
class BinaryClassifier(nn.Module):
    def __init__(self, x_dim=784):
        super().__init__()
        self.flatten = nn.Flatten()
        self.W = nn.Linear(x_dim, 1)

    def forward(self, x):
        x = self.flatten(x)
        return self.W(x)

### Creating the model

In [ ]:
model = BinaryClassifier()

## Training the Model

### Put the model onto the right device

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")

### Define the loss function and optimizer, and train the model

We'll talk about the `loss_fn` and the `optimizer` in the next lecture. 

The rest of this cell is what we call the _training loop_. All machine learning systems go through a training loop that is very similar to this one: read some data, move it to the GPU, run the model on the data to get predictions, evaluate those predictions, and then update the model based on the quality of the predictions.

Once we explain what `loss_fn` and `optimizer` are doing, we'll be able to walk through this loop and understand much more about what's going on.

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

EPOCHS = 1
for epoch in range(EPOCHS):
    for X,y in train_loader:
        X, y = X.to(device), y.to(device, dtype=torch.float)
        optimizer.zero_grad()
        logits = model(X)
        loss = loss_fn(logits.squeeze(-1), y)
        print(f"Epoch {epoch} Loss = {loss.item()}")
        loss.backward()
        optimizer.step()

### Estimate the accuracy

In [ ]:
correct = 0
total = 0

for X,y in test_loader:
    logits = model(X)
    y_out = (logits >= 0.0).squeeze(1)
    correct += (y == y_out).sum()
    total += len(logits)

print(f"Number correct = {correct} out of {total}")

## Visualize The Model's Output

### First visualize the data with correct labels

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from sklearn.decomposition import PCA

train_images = torch.cat([
    images.flatten(start_dim=1)
    for images, _ in DataLoader(train_ds, batch_size=512, shuffle=False)
]).numpy()
test_batches = list(DataLoader(test_ds, batch_size=512, shuffle=False))
test_images = torch.cat([images.flatten(start_dim=1) for images, _ in test_batches]).numpy()
labels = torch.cat([labels for _, labels in test_batches]).numpy()

pca = PCA(n_components=2, svd_solver="randomized", random_state=0)
pca.fit(train_images)
points = pca.transform(test_images)

fig, ax = plt.subplots(figsize=(10, 8))
for digit in np.unique(labels):
    mask = labels == digit
    ax.scatter(points[mask, 0], points[mask, 1], s=10, alpha=0.6,
               label=f"Digit {digit}")
ax.set(
    xlabel=f"Principal component 1 ({pca.explained_variance_ratio_[0]:.1%} variance)",
    ylabel=f"Principal component 2 ({pca.explained_variance_ratio_[1]:.1%} variance)",
    title="Binary MNIST: test images projected onto training PCA components",
)
ax.legend(title="True label")
plt.show()

### Then visualize how the model classifies points

In [ ]:
# Match the image shape, device, and dtype used by the model.
image_shape = test_batches[0][0].shape[1:]  # Usually (1, 28, 28)
param = next(model.parameters())

# 1. Create a grid in the 2D PCA plane.
grid_size = 250  # Number of grid points along each axis

x_min, x_max = points[:, 0].min() - 1, points[:, 0].max() + 1
y_min, y_max = points[:, 1].min() - 1, points[:, 1].max() + 1

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, grid_size),
    np.linspace(y_min, y_max, grid_size),
)
grid_points = np.column_stack([xx.ravel(), yy.ravel()])

# 2. Reconstruct images from the grid and classify them in batches.
was_training = model.training
model.eval()

predictions = []

with torch.no_grad():
    for start in range(0, len(grid_points), 512):
        grid_batch = grid_points[start:start + 512]

        # [batch, 2] -> [batch, 784]
        reconstructed = pca.inverse_transform(grid_batch)

        # Restore the model's expected image shape.
        images = torch.as_tensor(
            reconstructed,
            dtype=param.dtype,
            device=param.device,
        ).reshape(-1, *image_shape)

        logits = model(images).reshape(-1)  # One logit per image

        # Logit >= 0 corresponds to sigmoid(logit) >= 0.5.
        predictions.append((logits >= 0).cpu().numpy())

model.train(was_training)

Z = np.concatenate(predictions).reshape(xx.shape)

# 3. Colour the background by the model's predicted class.
fig, ax = plt.subplots(figsize=(10, 8))

ax.pcolormesh(
    xx, yy, Z,
    cmap="cool",
    vmin=0, vmax=1,
    shading="auto",
    alpha=0.25,
)

# 4. Overlay the test images, coloured by their true labels.
for cls in np.unique(labels):
    mask = labels == cls
    ax.scatter(
        points[mask, 0], points[mask, 1],
        c=labels[mask],
        cmap="cool",
        vmin=0, vmax=1,
        s=12,
        edgecolors="black",
        linewidths=0.25,
        label=f"Class {int(cls)}",
    )

ax.set(
    xlabel=f"Principal component 1 ({pca.explained_variance_ratio_[0]:.1%} variance)",
    ylabel=f"Principal component 2 ({pca.explained_variance_ratio_[1]:.1%} variance)",
    title="Binary MNIST: model decisions on the PCA plane",
)
ax.legend(title="True label")
plt.show()